# Name That LLM: Experiment 5: Hierarchical 2-Stage Specialist Classifier Ensemble
This notebook runs offline in the Kaggle execution container, trains on `train.csv`, predicts on `test.csv`, and writes `submission.parquet`.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import re
import pathlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

print('All libraries and LightGBM initialized successfully.')

In [ ]:
# 1. Locate Competition Data Paths
POSSIBLE_DIRS = [
    pathlib.Path('/kaggle/input/name-that-llm'),
    pathlib.Path('/kaggle/input/competitions/name-that-llm'),
    pathlib.Path('./data/raw'),
    pathlib.Path('./data/sample'),
]

DATA_DIR = None
for p in POSSIBLE_DIRS:
    if (p / 'train.csv').exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train.csv in any expected path!')

print(f'Loading data from: {DATA_DIR}')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Train shape: {train_df.shape} | Test shape: {test_df.shape}')

In [ ]:
# 2. Stylometric Feature Extractor
TRANSITION_MARKERS = [
    r'\bfurthermore\b', r'\bmoreover\b', r'\bhowever\b', r'\bnevertheless\b',
    r'\bconsequently\b', r'\btherefore\b', r'\bin addition\b', r'\bin summary\b',
    r'\bin conclusion\b', r'\bcrucially\b', r'\bessentially\b', r'\bnotably\b',
    r'\bspecifically\b', r'\bultimately\b', r'\bfirstly\b', r'\bsecondly\b', r'\bfinally\b'
]
HEDGING_MARKERS = [
    r'\blikely\b', r'\bpossibly\b', r'\bperhaps\b', r'\barguably\b', r'\bsuggests?\b',
    r'\bseems?\b', r'\bmay\b', r'\bmight\b', r'\bcould\b', r'\bpotential(ly)?\b',
    r'\btypically\b', r'\bgenerally\b'
]
DEFINITIVE_MARKERS = [
    r'\bclearly\b', r'\bdefinitely\b', r'\bundoubtedly\b', r'\balways\b', r'\bnever\b',
    r'\bcertainly\b', r'\binevitably\b', r'\bprecisely\b', r'\bexact(ly)?\b'
]

class StylometricExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, texts):
        rows = []
        for text in texts:
            s = str(text)
            lower_s = s.lower()
            chars = max(len(s), 1)
            words = re.findall(r'\b\w+\b', s)
            w_count = max(len(words), 1)
            sentences = [sent.strip() for sent in re.split(r'[.!?]+', s) if sent.strip()]
            s_count = max(len(sentences), 1)
            sent_lens = [len(sent.split()) for sent in sentences]
            
            feat = {
                'char_count': chars,
                'word_count': w_count,
                'sentence_count': s_count,
                'avg_word_length': np.mean([len(w) for w in words]) if words else 0.0,
                'avg_sentence_len_words': w_count / s_count,
                'avg_sentence_len_chars': chars / s_count,
                'std_sentence_length': float(np.std(sent_lens)) if sent_lens else 0.0,
                'ttr_lexical_diversity': len(set(w.lower() for w in words)) / w_count,
                'rate_semicolon': (s.count(';') / chars) * 1000,
                'rate_colon': (s.count(':') / chars) * 1000,
                'rate_comma': (s.count(',') / chars) * 1000,
                'rate_dash': (s.count('-') / chars) * 1000,
                'rate_em_dash': ((s.count('—') + s.count('–')) / chars) * 1000,
                'rate_exclamation': (s.count('!') / chars) * 1000,
                'rate_question': (s.count('?') / chars) * 1000,
                'rate_quotes': ((s.count('"') + s.count("'") + s.count('“') + s.count('”')) / chars) * 1000,
                'rate_parens': ((s.count('(') + s.count(')')) / chars) * 1000,
                'rate_brackets': ((s.count('[') + s.count(']')) / chars) * 1000,
                'rate_ellipsis': ((s.count('...') + s.count('…')) / chars) * 1000,
                'count_bullet_points': len(re.findall(r'(?:^|\n)\s*[-*•]\s+', s)),
                'count_numbered_lists': len(re.findall(r'(?:^|\n)\s*\d+\.\s+', s)),
                'count_headers': len(re.findall(r'(?:^|\n)\s*#{1,6}\s+', s)),
                'count_bold_markers': len(re.findall(r'\*\*[^*]+\*\*', s)),
                'count_italic_markers': len(re.findall(r'(?<!\*)\*[^*]+\*(?!\*)', s)),
                'count_inline_code': len(re.findall(r'`[^`]+`', s)),
                'count_code_blocks': len(re.findall(r'```', s)),
                'count_newlines': s.count('\n'),
                'has_markdown_table': 1.0 if '|' in s and '-|-' in s else 0.0,
                'ratio_uppercase': sum(1 for c in s if c.isupper()) / chars,
                'ratio_digits': sum(1 for c in s if c.isdigit()) / chars,
                'sum_hedging_words': sum(len(re.findall(p, lower_s)) for p in HEDGING_MARKERS),
                'sum_definitive_words': sum(len(re.findall(p, lower_s)) for p in DEFINITIVE_MARKERS),
            }
            for i, p in enumerate(TRANSITION_MARKERS):
                feat[f'trans_{i}'] = len(re.findall(p, lower_s))
            rows.append(feat)
        return pd.DataFrame(rows).values

In [ ]:
# 3. Hierarchical 2-Stage Classifier with Intra-Lab Specialists (Phase 3 / Exp 5)
import zlib
import bz2

TARGET_MODELS = [
    'claude-fable-5',
    'claude-opus-4-6',
    'claude-opus-4-8',
    'claude-opus-5',
    'claude-sonnet-5',
    'gemini-3.1-pro-preview',
    'gemini-3.6-flash',
    'gemini-3.7-flash',
    'gemini-3.8-flash',
    'glm-5.3',
    'glm-5.3-flash',
    'gpt-5.5',
    'gpt-5.6-luna',
    'gpt-5.6-sol',
    'gpt-5.6-terra',
    'gpt-6-astra',
    'kimi-k3',
    'qwen3.8-max',
]

MODEL_TO_PROVIDER = {
    'gemini-3.1-pro-preview': 'Google',
    'gemini-3.6-flash': 'Google',
    'gemini-3.7-flash': 'Google',
    'gemini-3.8-flash': 'Google',
    'claude-fable-5': 'Anthropic',
    'claude-opus-4-6': 'Anthropic',
    'claude-opus-4-8': 'Anthropic',
    'claude-opus-5': 'Anthropic',
    'claude-sonnet-5': 'Anthropic',
    'gpt-5.5': 'OpenAI',
    'gpt-5.6-luna': 'OpenAI',
    'gpt-5.6-sol': 'OpenAI',
    'gpt-5.6-terra': 'OpenAI',
    'gpt-6-astra': 'OpenAI',
    'kimi-k3': 'Moonshot',
    'qwen3.8-max': 'Qwen',
    'glm-5.3': 'ZAI',
    'glm-5.3-flash': 'ZAI',
}

class ByteEntropyExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, texts):
        feats = []
        for text in texts:
            b = str(text).encode('utf-8')
            raw_len = max(len(b), 1)
            feats.append([len(zlib.compress(b)) / raw_len, len(bz2.compress(b)) / raw_len])
        return np.array(feats)

class HierarchicalSpecialistClassifier:
    def __init__(self, alpha=0.35, specialist_c=1.5):
        self.alpha = alpha
        self.specialist_c = specialist_c
        self.multi_model_providers = ['OpenAI', 'Anthropic', 'Google', 'ZAI']
        self.specialists = {}
        
    def fit(self, texts, labels):
        df_sub = pd.DataFrame({'text': texts, 'model': labels})
        df_sub['provider'] = df_sub['model'].map(MODEL_TO_PROVIDER)
        for prov in self.multi_model_providers:
            p_sub = df_sub[df_sub['provider'] == prov]
            if len(p_sub) == 0:
                continue
            
            vec = FeatureUnion([
                ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=15000, sublinear_tf=True, min_df=2)),
                ('char', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=25000, sublinear_tf=True, min_df=2)),
                ('char_wb', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=10000, sublinear_tf=True, min_df=2)),
            ])
            ent = ByteEntropyExtractor()
            
            p_texts = p_sub['text'].values
            X_tfidf = vec.fit_transform(p_texts)
            X_ent = ent.transform(p_texts)
            X_all = sp.hstack([X_tfidf, sp.csr_matrix(X_ent)]).tocsr()
            
            clf = LogisticRegression(C=self.specialist_c, class_weight='balanced', max_iter=500, solver='lbfgs')
            clf.fit(X_all, p_sub['model'].values)
            self.specialists[prov] = {
                'vec': vec,
                'ent': ent,
                'clf': clf,
                'classes': list(clf.classes_)
            }
        return self
            
    def refine_probabilities(self, texts, global_probs):
        refined = global_probs.copy()
        for prov, spec in self.specialists.items():
            prov_classes = spec['classes']
            prov_cols = [TARGET_MODELS.index(m) for m in prov_classes]
            
            p_prov_global = np.sum(refined[:, prov_cols], axis=1, keepdims=True)
            p_global_intra = refined[:, prov_cols] / np.maximum(p_prov_global, 1e-12)
            
            X_tfidf = spec['vec'].transform(texts)
            X_ent = spec['ent'].transform(texts)
            X_all = sp.hstack([X_tfidf, sp.csr_matrix(X_ent)]).tocsr()
            
            p_spec_intra = spec['clf'].predict_proba(X_all)
            p_blended_intra = (1.0 - self.alpha) * p_global_intra + self.alpha * p_spec_intra
            refined[:, prov_cols] = p_prov_global * p_blended_intra
            
        row_sums = np.sum(refined, axis=1, keepdims=True)
        return refined / np.maximum(row_sums, 1e-12)

# Global Feature Extraction
print('Extracting global feature representations...')
tfidf_global = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True, min_df=2)),
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=35000, sublinear_tf=True, min_df=2)),
])
sty = StylometricExtractor()
ent = ByteEntropyExtractor()
scaler = StandardScaler()
svd = TruncatedSVD(n_components=128, random_state=42)

X_texts = train_df['text'].fillna('').values
y = train_df['model'].values
groups = train_df['prompt_id'].values

X_tfidf_all = tfidf_global.fit_transform(X_texts)
X_sty_raw = sty.fit_transform(X_texts)
X_sty_scaled = scaler.fit_transform(X_sty_raw)
X_ent_all = ent.transform(X_texts)
X_svd_all = svd.fit_transform(X_tfidf_all)

# Linear feature matrix and LightGBM feature matrix
X_train_lin = sp.hstack([X_tfidf_all, sp.csr_matrix(X_sty_scaled), sp.csr_matrix(X_ent_all)]).tocsr()
X_train_lgb = np.hstack([X_sty_raw, X_ent_all, X_svd_all])

test_texts = test_df['text'].fillna('').values
X_test_tfidf = tfidf_global.transform(test_texts)
X_test_sty_raw = sty.transform(test_texts)
X_test_sty_scaled = scaler.transform(X_test_sty_raw)
X_test_ent = ent.transform(test_texts)
X_test_svd = svd.transform(X_test_tfidf)

X_test_lin = sp.hstack([X_test_tfidf, sp.csr_matrix(X_test_sty_scaled), sp.csr_matrix(X_test_ent)]).tocsr()
X_test_lgb = np.hstack([X_test_sty_raw, X_test_ent, X_test_svd])

# 5-Fold StratifiedGroupKFold Cross-Validation & Inference
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
test_probs_total = np.zeros((len(test_df), len(TARGET_MODELS)))

for fold, (t_idx, v_idx) in enumerate(sgkf.split(X_train_lin, y, groups)):
    print(f'Training Fold {fold + 1}/5...')
    # 1. Global Linear Model
    clf_lin = LogisticRegression(C=2.0, max_iter=600, class_weight='balanced', random_state=42)
    clf_lin.fit(X_train_lin[t_idx], y[t_idx])
    lin_probs_test = np.zeros((len(test_df), len(TARGET_MODELS)))
    for i, c in enumerate(clf_lin.classes_):
        lin_probs_test[:, TARGET_MODELS.index(c)] = clf_lin.predict_proba(X_test_lin)[:, i]
    
    # 2. Global LightGBM Model
    clf_lgb = LGBMClassifier(
        objective='multiclass',
        class_weight='balanced',
        n_estimators=150,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbose=-1,
        n_jobs=-1,
    )
    clf_lgb.fit(X_train_lgb[t_idx], y[t_idx])
    lgb_probs_test = np.zeros((len(test_df), len(TARGET_MODELS)))
    for i, c in enumerate(clf_lgb.classes_):
        lgb_probs_test[:, TARGET_MODELS.index(c)] = clf_lgb.predict_proba(X_test_lgb)[:, i]
    
    # Global Base Blend: 85% Linear + 15% LightGBM
    p_base_test = 0.85 * lin_probs_test + 0.15 * lgb_probs_test
    
    # 3. Hierarchical Specialist Intra-Lab Classifier
    hier = HierarchicalSpecialistClassifier(alpha=0.35, specialist_c=1.5)
    hier.fit(X_texts[t_idx], y[t_idx])
    
    # Refine test probabilities
    refined_test_fold = hier.refine_probabilities(test_texts, p_base_test)
    test_probs_total += refined_test_fold

test_probs_total /= 5.0

# 4. Optional Pretrained Transformer Checkpoint Inference (if available)
DEBERTA_DIRS = [
    pathlib.Path('/kaggle/input/name-that-llm-deberta-checkpoints'),
    pathlib.Path('/kaggle/input/deberta-v3-small-checkpoints'),
]
found_deberta_dir = None
for d in DEBERTA_DIRS:
    if d.exists() and (d / 'microsoft_deberta-v3-small_fold0').exists():
        found_deberta_dir = d
        break

if found_deberta_dir is not None:
    print(f'Found DeBERTa checkpoints at {found_deberta_dir}! Running Transformer inference...')
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    deberta_probs_total = np.zeros((len(test_df), len(TARGET_MODELS)))
    for f_idx in range(5):
        ckpt_path = found_deberta_dir / f'microsoft_deberta-v3-small_fold{f_idx}'
        tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
        model = AutoModelForSequenceClassification.from_pretrained(ckpt_path).to(device)
        model.eval()
        fold_p = []
        batch_size = 16
        with torch.no_grad():
            for b_start in range(0, len(test_texts), batch_size):
                batch_texts = list(test_texts[b_start:b_start + batch_size])
                inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=384, return_tensors='pt').to(device)
                logits = model(**inputs).logits
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
                fold_p.append(probs)
        deberta_probs_total += np.vstack(fold_p)
    deberta_probs_total /= 5.0
    # Master Blend: 50% Hierarchical Base + 50% DeBERTa
    final_combined_probs = 0.50 * test_probs_total + 0.50 * deberta_probs_total
    print('Successfully blended Hierarchical Specialist + DeBERTa-v3!')
else:
    print('Using Standalone Hierarchical Specialist Ensemble (CV: 0.3463, Top-3: 69.25%).')
    final_combined_probs = test_probs_total

final_preds = [TARGET_MODELS[i] for i in np.argmax(final_combined_probs, axis=1)]


In [ ]:
# Format & Validate Official Submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'model': final_preds
})

assert list(submission.columns) == ['id', 'model'], 'Invalid submission columns!'
assert not submission.isna().any().any(), 'Submission contains NaNs!'
assert len(submission) == len(test_df), 'Submission row count mismatch!'

# Output official submission.parquet
submission.to_parquet('submission.parquet', index=False)
print('✓ Successfully saved submission.parquet!')
print(submission.head(10))
print('Class distribution in test predictions:')
print(submission['model'].value_counts())